In [ ]:
%%configure  

{ 
    "vCores": 
    { 
        "parameterName": "pipelinecore", 
        "defaultValue": 2 
    }
}

In [ ]:
# dbt-fabric talks to the Fabric Warehouse over ODBC (pyodbc). The Fabric runtime ships
# the Microsoft ODBC driver; we only need the adapter + azure-identity for token auth.
!pip install -q dbt-fabric azure-identity --upgrade
notebookutils.session.restartPython()

In [ ]:
import os
try:
    import notebookutils
    vl             = notebookutils.variableLibrary.getLibrary("deploy_config_dwh")
    workspace_id   = vl.workspace_id
    lakehouse_name = vl.lakehouse_name
    warehouse_name = vl.warehouse_name
    download_limit = vl.download_limit
    process_limit  = vl.process_limit
    lakehouse_id   = notebookutils.lakehouse.get(lakehouse_name).get("id")
    dbt_target     = "dev"

    # Resolve the Warehouse SQL endpoint (server) + id via the Fabric REST API.
    import requests
    api_token = notebookutils.credentials.getToken("pbi")
    resp = requests.get(
        f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/warehouses",
        headers={"Authorization": f"Bearer {api_token}"},
        timeout=60,
    )
    resp.raise_for_status()
    wh = next(w for w in resp.json()["value"] if w["displayName"] == warehouse_name)
    dwh_server = wh["properties"]["connectionString"]
    dwh_name   = warehouse_name

    # Access token for the Warehouse SQL endpoint, handed to dbt-fabric.
    db_token = None
    for audience in ("https://database.windows.net/.default",
                     "https://analysis.windows.net/powerbi/api/.default", "pbi"):
        try:
            db_token = notebookutils.credentials.getToken(audience)
            if db_token:
                break
        except Exception:
            continue

    # Mount the lakehouse (cache off) so the dbt project under Files/dbt is on a local path.
    notebookutils.fs.mount(
        f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}",
        "/lh",
        {"fileCacheTimeout": 0},
    )
    mount_path = notebookutils.fs.getMountPath("/lh")
    dbt_path   = f"{mount_path}/Files/dbt"
    storage_token  = notebookutils.credentials.getToken("storage")
except ModuleNotFoundError:
    # Local / CI dev: deploy_config.yml local section + Azure CLI identity.
    from azure.identity import AzureCliCredential
    import yaml
    from pathlib import Path
    _root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "deploy_config.yml").exists()), None)
    if _root is None:
        raise FileNotFoundError("deploy_config.yml not found in cwd or any parent")
    _all   = yaml.safe_load((_root / "deploy_config.yml").read_text())
    _cfg   = {**_all.get("defaults", {}), **_all["local"]}
    workspace_id   = _cfg["ws"]
    lakehouse_id   = _cfg["lakehouse"]
    lakehouse_name = _cfg["lakehouse_name"]
    warehouse_name = _cfg["warehouse_name"]
    download_limit = _cfg["download_limit"]
    process_limit  = _cfg["process_limit"]
    dbt_path       = _cfg["dbt_path"]
    dwh_server     = os.environ["FABRIC_DWH_SERVER"]
    dwh_name       = os.environ.get("FABRIC_DWH_NAME", warehouse_name)
    db_token       = None  # local uses authentication=CLI
    storage_token  = AzureCliCredential().get_token("https://storage.azure.com/.default").token
    dbt_target     = "dev"
os.environ["download_limit"] = str(download_limit)
os.environ["process_limit"]  = str(process_limit)

In [ ]:
# OPENROWSET reads the raw CSVs + archive log under the lakehouse Files using the
# warehouse identity, so we only need FILES_PATH for the model paths.
os.environ["FILES_PATH"]        = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Files"
os.environ["FABRIC_DWH_SERVER"] = dwh_server
os.environ["FABRIC_DWH_NAME"]   = dwh_name
if db_token:
    # In Fabric: pass the SQL-endpoint access token to dbt-fabric.
    os.environ["FABRIC_AUTH"]         = "ActiveDirectoryAccessToken"
    os.environ["FABRIC_ACCESS_TOKEN"] = db_token
else:
    # Locally / CI: use the logged-in az identity.
    os.environ["FABRIC_AUTH"] = "CLI"

In [ ]:
# Land AEMO files BEFORE dbt: dbt-fabric can't run Python models, so the kept duckrun
# downloader (dbt/landing/stg_csv_archive_log.py) runs here as a standalone DuckDB step.
# It writes gzip CSVs to Files/csv/** + the watermark Files/csv_archive_log.parquet that
# the T-SQL models then read via OPENROWSET. DuckDB is treated as a fancy pandas here.
import duckdb, importlib.util

con = duckdb.connect()
con.sql("INSTALL azure; LOAD azure;")
# 'curl' transport avoids the Azure SDK's SSL CA cert error against OneLake.
con.sql("SET GLOBAL azure_transport_option_type='curl';")
con.sql(
    "CREATE OR REPLACE SECRET onelake "
    "(TYPE azure, PROVIDER access_token, ACCESS_TOKEN '" + storage_token + "')"
)

_spec = importlib.util.spec_from_file_location(
    "aemo_downloader", f"{dbt_path}/landing/stg_csv_archive_log.py"
)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)

class _DbtStub:  # dbt.config(...) is a no-op outside dbt; session is the duckdb connection
    def config(self, **kwargs):
        pass

_mod.model(_DbtStub(), con)
con.close()

In [ ]:
from dbt.cli.main import dbtRunner
os.chdir(dbt_path)
dbt = dbtRunner()
import datetime
# One timestamped log folder per run, beside the dbt project (NOT inside it) so logs survive a crash.
_log_dir = f"{os.path.dirname(dbt_path)}/dbt_runs_logs/run-{datetime.datetime.now():%Y%m%dT%H%M%S}"
base = ["--target", dbt_target, "--profiles-dir", ".",
        "--log-path", _log_dir, "--target-path", "/tmp/dbt_target"]

# Files already landed by the DuckDB downloader cell above; dbt reads them via OPENROWSET.

# 1. New daily files pending? check_new_daily "fails" when yes -- checked BEFORE fct_scada ingests them.
new_daily = not dbt.invoke(["run-operation", "check_new_daily", *base]).success
print(f"new_daily = {new_daily}")

# 2. Build everything except the summary (retry failures once).
#    No new daily landed -> the daily fct_price/fct_scada have nothing to do, so exclude them.
daily_ex = [] if new_daily else ["--exclude", "fct_price", "--exclude", "fct_scada"]
result = dbt.invoke(["run", "--exclude", "fct_summary", *daily_ex, *base])
if not result.success:
    print("dbt run had failures -- retrying failed models once...")
    _ = dbt.invoke(["retry", *base])

# 3. Summary: overwrite when a new daily landed, else append intraday.
summary = ["run", "--select", "fct_summary", *base] + (["--full-refresh"] if new_daily else [])
_ = dbt.invoke(summary)

# 4. Tests.
_ = dbt.invoke(["test", *base])